# glass0: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('17_glass0_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,72.732,64.789,69.718,61.429,87.488,74.458,70.209,85.199,68.761
smote,79.919,72.650,64.444,84.286,76.293,80.289,71.221,86.871,76.285
bsmote,79.637,71.857,63.361,84.286,76.010,80.148,67.310,85.279,75.747
ros,79.904,72.240,65.144,82.143,78.436,80.289,71.821,86.014,76.072
adasyn,80.832,73.333,62.787,88.571,73.929,81.250,70.018,86.675,77.082
casb,79.092,71.454,63.343,82.857,75.985,79.421,70.635,86.430,75.273
casb_relaxed,80.247,72.673,64.993,83.571,77.734,80.653,71.782,86.535,76.460
casb_edited,77.685,69.635,58.481,86.429,70.123,78.276,68.522,86.656,73.660
casb_ros,79.280,71.686,63.685,82.857,76.330,79.594,70.473,86.206,75.483


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,77.666,69.718,64.798,75.714,79.901,77.808,78.118,87.964,73.692
smote,79.466,71.611,63.515,82.857,76.761,79.809,78.634,88.662,75.539
bsmote,79.658,72.170,61.085,89.286,71.527,80.406,80.487,89.714,75.914
ros,77.254,68.984,61.334,79.286,75.677,77.482,78.099,87.849,73.119
adasyn,82.517,75.083,62.818,93.571,72.906,83.239,80.657,89.860,78.800
casb,80.002,72.289,64.330,82.857,77.438,80.148,78.959,88.260,76.145
casb_relaxed,80.770,73.179,63.698,86.429,75.690,81.059,80.310,89.304,76.974
casb_edited,78.723,70.952,59.760,87.857,70.813,79.335,69.380,86.635,74.837
casb_ros,77.624,69.585,61.902,80.000,75.677,77.839,77.921,87.873,73.605


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,83.457,78.542,80.995,76.429,91.355,83.892,86.979,93.406,80.999
smote,87.054,81.691,77.697,87.143,87.500,87.321,86.066,93.038,84.372
bsmote,85.464,79.930,76.764,84.286,87.143,85.714,85.969,92.514,82.697
ros,85.956,80.707,78.795,83.571,88.904,86.238,86.418,93.097,83.332
adasyn,86.999,81.313,76.028,87.857,86.453,87.155,87.133,92.883,84.156
casb,85.036,79.502,77.597,82.143,88.522,85.333,87.170,93.551,82.269
casb_relaxed,86.123,80.843,78.810,83.571,89.236,86.404,86.422,93.204,83.483
casb_edited,84.387,77.743,69.559,88.571,80.554,84.563,84.878,92.317,81.065
casb_ros,86.717,81.857,80.553,83.571,90.296,86.933,87.303,93.751,84.287


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,67.602,58.459,64.859,54.286,85.443,69.865,66.735,83.102,63.030
smote,71.634,62.661,54.503,75.000,69.409,72.204,66.374,83.001,67.148
bsmote,73.412,64.793,55.216,79.286,68.744,74.015,64.064,82.239,69.102
ros,73.070,64.732,59.787,72.857,74.631,73.744,66.589,83.055,68.901
adasyn,70.635,61.850,52.121,77.143,65.591,71.367,65.777,83.055,66.242
casb,72.762,63.966,55.294,76.429,69.766,73.097,66.907,83.230,68.364
casb_relaxed,72.768,63.837,55.506,75.714,70.480,73.097,66.647,83.103,68.303
casb_edited,74.148,66.033,56.858,80.000,69.446,74.723,66.909,82.972,70.091
casb_ros,71.410,62.620,56.976,71.429,72.525,71.977,67.040,83.157,67.015


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,75.364,67.877,70.092,66.964,86.047,76.506,75.510,87.418,71.621
smote,79.518,72.153,65.040,82.321,77.491,79.906,75.574,87.893,75.836
bsmote,79.543,72.188,64.106,84.286,75.856,80.071,74.458,87.436,75.865
ros,79.046,71.666,66.265,79.464,79.412,79.438,75.732,87.504,75.356
adasyn,80.246,72.895,63.439,86.786,74.720,80.753,75.896,88.118,76.570
casb,79.223,71.803,65.141,81.071,77.928,79.500,75.918,87.868,75.513
casb_relaxed,79.977,72.633,65.752,82.321,78.285,80.303,76.290,88.036,76.305
casb_edited,78.736,71.091,61.164,85.714,72.734,79.224,72.422,87.145,74.913
casb_ros,78.758,71.437,65.779,79.464,78.707,79.086,75.684,87.747,75.097


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,80.247,adasyn,80.832,-0.585,LOSS
1,f1,casb_relaxed,72.673,adasyn,73.333,-0.659,LOSS
2,precision,casb_relaxed,64.993,ros,65.144,-0.151,LOSS
3,recall,casb_edited,86.429,adasyn,88.571,-2.143,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,80.770,adasyn,82.517,-1.747,LOSS
1,f1,casb_relaxed,73.179,adasyn,75.083,-1.904,LOSS
2,precision,casb,64.330,smote,63.515,0.815,WIN
3,recall,casb_edited,87.857,adasyn,93.571,-5.714,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,86.717,smote,87.054,-0.338,LOSS
1,f1,casb_ros,81.857,smote,81.691,0.166,WIN
2,precision,casb_ros,80.553,ros,78.795,1.758,WIN
3,recall,casb_edited,88.571,adasyn,87.857,0.714,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,79.977,adasyn,80.246,-0.269,LOSS
1,f1,casb_relaxed,72.633,adasyn,72.895,-0.262,LOSS
2,precision,casb_ros,65.779,ros,66.265,-0.486,LOSS
3,recall,casb_edited,85.714,adasyn,86.786,-1.071,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.607831,0.500000,0.600000,0.428571,0.862069,0.645320,0.733630,0.881773,0.553915,None
1,42,1,svm,smote,0.75,0.876626,0.827586,0.800000,0.857143,0.896552,0.876847,0.665198,0.891626,0.852106,None
2,42,1,svm,bsmote,1.00,0.850963,0.777778,0.636364,1.000000,0.724138,0.862069,0.694445,0.903941,0.814370,None
3,42,1,svm,ros,1.00,0.890564,0.823529,0.700000,1.000000,0.793103,0.896552,0.688896,0.901478,0.857046,None
4,42,1,svm,adasyn,0.75,0.894703,0.838710,0.764706,0.928571,0.862069,0.895320,0.676659,0.901478,0.866706,None
5,42,1,svm,casb,0.75,0.859602,0.800000,0.750000,0.857143,0.862069,0.859606,0.755478,0.903941,0.829801,None
6,42,1,svm,casb_relaxed,1.00,0.870988,0.800000,0.666667,1.000000,0.758621,0.879310,0.694020,0.901478,0.835494,None
7,42,1,svm,casb_edited,0.75,0.870988,0.800000,0.666667,1.000000,0.758621,0.879310,0.690252,0.901478,0.835494,None
8,42,1,svm,casb_ros,0.75,0.876626,0.827586,0.800000,0.857143,0.896552,0.876847,0.752443,0.901478,0.852106,None
9,42,1,knn,original,0.50,0.859602,0.800000,0.750000,0.857143,0.862069,0.859606,0.879314,0.918719,0.829801,None
